In [18]:
import networkx as nx
import numpy as np
import copy
import config as cfg
import helpers
from data_acquirer import get_data, gen_data

In [3]:
method = 2
random_data = True
total_robots = 5
total_tasks = 50
data_name = 'data4'

In [4]:
class Path:
    def __init__(self,depot):
        self.tasks =[depot]
        self.total_time = 0
        self.travel_time = 0
        pass
    
    def insert(self, task):
        self.tasks.append(task)
        self.update_travel_time(task)
        return
    
    def update_total_time(self, time):
        self.total_time = time
        return
    
    def update_travel_time(self,task):
        last_task = self.get_last_task()
        self.travel_time += helpers.cal_dist_task(last_task, task)
#         return travel_time
    
    def get_insert_cost(self, task, c_travel, c_operate):
        last_task = self.get_last_task()
        cost_if_insert = self.total_time + c_travel[last_task.index][task.index] + c_operate[task.index]
        return cost_if_insert
    
    def get_last_task(self):
        return self.tasks[-1]
    
    def __repr__(self):
            return f"Path {[task.index for task in self.tasks]}"

In [5]:
class Solution:
    def __init__(self, paths, depot, tasks):
        self.paths = paths
        self.depot = depot
        self.tasks = tasks
        
    def __repr__(self):
        return f"Paths {self.paths}"

In [6]:
is_depot_task = True
if random_data:
    raw = gen_data(total_robots, total_tasks, False, is_depot_task)
else:
    raw = get_data(data_name,'csv', is_depot_task)

In [7]:
raw.tasks

[Task 1,[ 27.18160715 180.76922304],[137.7004643  265.00011245],2,
 Task 2,[359.96254948 106.05442617],[436.21472239  16.24776521],2,
 Task 3,[358.88684374 432.8575171 ],[403.53974112 105.27529133],1,
 Task 4,[387.3228842   58.29954222],[181.83209001 411.71501716],4,
 Task 5,[380.25513696 236.73722134],[254.85765291 472.5192086 ],1,
 Task 6,[173.09748434 185.16753814],[364.15746919 175.25769225],1,
 Task 7,[332.81376891 182.16440028],[311.50930967 321.36244262],2,
 Task 8,[347.387103   163.01695775],[ 66.84839877 421.83137715],5,
 Task 9,[366.52014061 294.45439091],[ 81.99897047 241.76455475],3,
 Task 10,[319.74471416 222.99907376],[487.30331786 164.88296619],5,
 Task 11,[231.85572159 462.89866332],[169.17354163  52.67189981],2,
 Task 12,[134.41415063  11.95221103],[185.66628288  48.86543983],5,
 Task 13,[229.72221267 260.2482374 ],[342.28452145 212.64514765],4,
 Task 14,[330.76521689 275.92252363],[163.88922515 235.5967731 ],2,
 Task 15,[306.43201251 387.486912  ],[482.08515211 497.61

In [9]:
c_travel = helpers.create_task_dist_dict(raw.depot,raw.tasks)
c_operate = helpers.create_operate_dict(raw.depot,raw.tasks)
for i in range(len(c_travel)):
    c_travel[i].pop(i)

In [10]:
c_travel

{0: {1: 407.58932028359527,
  2: 66.5514846719577,
  3: 349.60442694581053,
  4: 48.43132087352503,
  5: 153.7650026370088,
  6: 268.9231098141721,
  7: 130.33033596191987,
  8: 106.51171939517822,
  9: 213.02285845177516,
  10: 169.67757770304434,
  11: 420.1691715366333,
  12: 300.1708119743023,
  13: 258.9896139712916,
  14: 208.6669397175639,
  15: 320.53977804067483,
  16: 400.36918167580797,
  17: 550.4764086675026,
  18: 110.47602508791431,
  19: 406.88763521597434,
  20: 492.1373237650302,
  21: 551.0995938091273,
  22: 214.57361998812527,
  23: 236.78515751019071,
  24: 306.01825210280884,
  25: 338.87026946700166,
  26: 278.59375573286025,
  27: 330.24254089790065,
  28: 121.12175540867831,
  29: 305.33199727892895,
  30: 76.49702635471165,
  31: 396.0581885669773,
  32: 291.23353590825224,
  33: 106.8615764020002,
  34: 395.9535438198592,
  35: 384.00573029212984,
  36: 169.17806964742476,
  37: 472.457829287981,
  38: 477.5229507340363,
  39: 352.7515639678655,
  40: 100.22

In [ ]:


uninserted = copy.deepcopy(raw.tasks) #j left
paths = [Path(raw.depot) for _ in range(total_robots)]
cur_sol = Solution(paths,raw.depot, raw.tasks)
best_obj = 0

idx_task_mapper = {task.index: task for task in raw.tasks}

C_max = 0
while len(uninserted) > 0:
    shortest_path_length = 9_999_999
    for path in cur_sol.paths:
        last_task = path.get_last_task()
        last_path_idx = last_task.index
        closest_idx = min(c_travel[last_path_idx],key = c_travel[last_path_idx].get)
        closest_task = idx_task_mapper[closest_idx]
        cost_if_insert = path.get_insert_cost(closest_task,c_travel,c_operate)
#         delta  = cost_if_insert - C_max
    
        if cost_if_insert < shortest_path_length:
            shortest_path_length = cost_if_insert
            best_path = path
            best_task = idx_task_mapper[closest_idx]
    
    print(f"BEST PATH: {best_path} \n BEST TASK: {best_task}")
    costs_if_insert = {path: path.get_insert_cost(best_task,c_travel, c_operate)}
#     print(f'c_travel {c_travel}')
    sorted_cost = list(np.sort(list(costs_if_insert.values())))
    shortest_to_insert_costs = sorted_cost[:best_task.n]
    inserted_paths_length = shortest_to_insert_costs[-1]
    for _ in range(best_task.n):
        cur_length = shortest_to_insert_costs.pop()
        for path, length in costs_if_insert.items():
            if length == cur_length:
                path.insert(best_task)
                path.update_total_time(inserted_paths_length)
                
    for i in range(len(c_travel)):
        try:
            c_travel[i].pop(best_task.index)
        except:
            pass
    
    uninserted.pop(best_task)
    # Remember to remove best_task from c_travel

In [212]:
some_list = [69,30]
some_list.remove(30)

In [215]:
c_travel

{0: {1: 205.38352145004262,
  2: 330.5746976357832,
  4: 331.73438131055724,
  5: 130.70201389018493,
  6: 462.8704755886657,
  7: 47.85753621443574,
  8: 78.32231292174166,
  9: 52.845607042614816,
  10: 325.36364876587385,
  11: 262.8153420847011,
  12: 57.57702143017476,
  13: 249.35772892950416,
  14: 284.33781087311513,
  15: 116.29907845615075,
  16: 61.67152891303426,
  17: 465.88003167169563,
  18: 198.01880061688348,
  19: 109.74602414035705,
  20: 193.62842549230788,
  21: 163.70810146587772,
  22: 346.3440490526702,
  23: 326.45634671616153,
  24: 215.90206256422377,
  25: 323.2796182333414,
  26: 93.07859848624383,
  27: 133.4058933576663,
  28: 112.96970997728421,
  29: 435.76092112922026,
  30: 59.81019313307948,
  31: 262.4646613622698,
  32: 313.75063612319644,
  33: 323.48226541547433,
  34: 50.70888624133648,
  35: 495.5499008544004,
  36: 431.46433961203803,
  37: 318.76514166617744,
  38: 286.5100539997024,
  39: 36.52352404945473,
  40: 98.04231675694426,
  41: 243

In [ ]:
# while len(uninserted) > 0:
#     cur_task = uninserted.pop()
#     deltas = {}
#     for path in paths:
#         cur_time = path.cost
#         travel_time = helpers.cal_dist_task(path.last_task(),cur_task)
#         process_time = cur_task.process_time
#         cost_if_insert = cur_time + travel_time + process_time
#         delta = cost_if_insert - best_obj
#         deltas[path] = delta

In [ ]:
def insert_single(solution, task):
    """
    Find a best spot to insert the `task` into `solution`
    
    Parameters
    ----------
    solution: Solution
    current solution
    task : Task
    current task want to insert to the solution, note that this must be a single task.
    """
    for path in soltion.paths: 
        for i in range(len(path)):
            temp_path = path.insert(i,task)
            

In [11]:
def paths_to_edges(paths: list): 
    """
    extract the edges to the directed graph G from a
    set of paths. More on section 2 in "Formal construct
    solution from paths 13-12-2023" document
    
    Parameters
    ----------
    paths : list
        list of tuples, every tuple is a path of every robot
        excluding the sink and the source (nessessary?)
           
    Returns
    -------
    edges : list
        list of edges, note that there may be duplications edges
        
    """
    edges = []
    G = nx.DiGraph()
    for path in paths:
        cur_len = len(path)
        edges.append((-1,path[0]))
        for i in range(cur_len-1):
            edges.append((path[i],path[i+1]))
        edges.append((path[-1],-99))
    return edges

In [49]:
def get_out_nodes(graph: nx.DiGraph):
    """
    Get a dictionary of out node from all nodes on a directed graph
    
    Parameters
    ----------
    graph : nx.DiGraph


    Returns
    -------
    out_nodes : dict
        `outnodes[1] = [2,3,69]` means that edges (1,2), (1,3), (1,69) in the graph.
    """
    out_nodes = {}
    nodes = list(graph.nodes())
    for node in nodes:
        out_nodes[node] = list(graph.successors(node))

    return out_nodes

In [ ]:
def paths_to_sol(paths, c_travel, c_operates, obj= "makespan"):
    """
    Construct solution from paths, consists of starting time
    of everynode and the makespan
    
    Parameters
    ----------
    paths : list
        see function `paths_to_edges()`
    c_travel : dict
        c_travel[0][4] is c_04
    c_operate: dict
        c_operate[5] is c_5
    
    Returns
    -------
    #TODO
    """
    G = nx.DiGraph()
    edges = paths_to_edges(paths)
    G.add_edges_from(edges)
    reversed_topo_order = list(reversed(list(nx.topological_sort(G))))
    out_nodes = get_out_nodes(G)

    z = {}
    z[-1] = 0
    for j in c_operate:
        z[j] = -np.infty
    z[-99] = -np.infty
    while reversed_topo_order is not None:
        cur_task = reversed_topo_order.pop()
        for succ in out_nodes[cur_task]:
            z_cur = z[cur_task] + c_operate[cur_task] + c_travel[cur_task][succ]
            if z[succ] < z_cur:
                z[succ] = z_cur
    return z
    

In [16]:
#trash
paths = [(1,2,5,6),(4,2,5,7),(8,9)]
G = nx.DiGraph()
edges = paths_to_edges(paths)
G.add_edges_from(edges)
list(nx.topological_sort(G))

[-1, 1, 4, 8, 2, 9, 5, 6, 7, -99]